# Multimodal Transformer–Cross-Attention + Double Machine Learning
## Indonesian Musicians: Platform Demand, Occupational Identity Transformation, and Quality of Working Life

**Purpose:** End-to-end, executable research prototype implementing the proposed workflow.

### Important data note
No musician dataset was attached to this chat. Therefore, the notebook first **creates a reproducible synthetic Indonesian musician database** with realistic heterogeneous variables, missing values, anomalies, occupational narratives, regional labels, and repeated observations. 

To use real data, replace the database-generation cell with your harmonised participant-level data while keeping the required conceptual fields.

### Implemented stages
1. Multi-source data acquisition/integration
2. Data quality enhancement
3. Platform Demand Index (PDI)
4. Sentence-Transformer + topic modelling occupational identity
5. **MUSICA-MOR** multimodal representation
6. Cross-attention demand–identity interaction
7. TabTransformer-style feature interaction + gated MLP QWL prediction
8. Deep Embedded Clustering + temporal Transformer identity trajectories
9. SHAP + Integrated Gradients + counterfactual explanation
10. **PIQ-Causal** Double Machine Learning + mediation
11. **INDO-MUS-GEN** Leave-Region-Out validation
12. Metrics, plots, saved outputs, and model artifacts

The notebook is designed to avoid data leakage: preprocessing, representation learning, and prediction models are fitted inside the appropriate training folds wherever practical.


In [ ]:
# ============================================================
# 0. INSTALLATION / IMPORTS
# ============================================================
# Run this cell in a fresh Colab/Jupyter environment if needed.
# The notebook has fallbacks for heavyweight NLP packages.

import sys, subprocess, importlib.util, os, json, random, warnings
warnings.filterwarnings("ignore")

def ensure_package(import_name, pip_name=None):
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name or import_name])

core_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "scipy": "scipy",
    "joblib": "joblib",
    "torch": "torch",
}
for imp, pipn in core_packages.items():
    ensure_package(imp, pipn)

# Optional packages. Failure is allowed because fallbacks are implemented.
optional = [
    ("sentence_transformers", "sentence-transformers"),
    ("shap", "shap"),
    ("captum", "captum"),
    ("bertopic", "bertopic"),
    ("umap", "umap-learn"),
    ("hdbscan", "hdbscan"),
]
for imp, pipn in optional:
    try:
        ensure_package(imp, pipn)
    except Exception as e:
        print(f"Optional package unavailable: {pipn} -> fallback will be used.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import ExtraTreesRegressor, ExtraTreesClassifier, IsolationForest, RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.neighbors import NearestNeighbors
from scipy.special import expit
import joblib

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

OUTPUT_DIR = Path("musica_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Environment ready. Output:", OUTPUT_DIR.resolve())


## 1. Research Configuration

The following configuration makes the notebook easy to adapt to a real dataset. The database schema explicitly separates **participant-level variables**, **regional identifiers**, **text**, and **time**.

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================

N_MUSICIANS = 1200
YEARS = [2021, 2022, 2023, 2024, 2025]
REGIONS = [
    "Sumatra", "Java", "Bali", "Kalimantan",
    "Sulawesi", "Nusa Tenggara", "Maluku-Papua"
]

PLATFORM_DEMAND_VARS = [
    "posting_frequency",
    "content_production_time",
    "audience_management",
    "algorithmic_pressure",
    "engagement_monitoring",
    "monetisation_pressure",
    "platform_dependency"
]

QWL_COMPONENTS = [
    "work_satisfaction",
    "occupational_autonomy",
    "work_life_balance",
    "economic_security",
    "career_fulfilment",
    "occupational_control",
    "perceived_work_pressure"
]

OCCUPATIONAL_TEXT_COL = "occupational_narrative"
REGION_COL = "region"
ID_COL = "musician_id"
TIME_COL = "year"
TARGET_COL = "qwl"

print("Regions:", REGIONS)
print("Platform-demand dimensions:", PLATFORM_DEMAND_VARS)
print("QWL dimensions:", QWL_COMPONENTS)


## 2. Stage 1 — Create a New Integrated Research Database

This is a **database-construction prototype**, not fabricated empirical evidence. It simulates separate sources and then links them using a pseudonymised musician ID.

For real research:
- load survey data;
- load platform-use data;
- load occupational/career data;
- load eligible digital-content aggregates;
- load regional data;
- perform ethical linkage only where a valid key exists;
- otherwise keep sources independent or aggregate them.

The synthetic generator intentionally creates correlations among platform pressure, identity, income, autonomy and QWL so that the complete workflow can be tested.

In [ ]:
# ============================================================
# 2. SYNTHETIC MULTI-SOURCE DATABASE GENERATOR
# ============================================================

rng = np.random.default_rng(SEED)

# Stable musician-level master table
musicians = pd.DataFrame({
    ID_COL: np.arange(1, N_MUSICIANS + 1),
    "region": rng.choice(REGIONS, N_MUSICIANS, p=[.14,.30,.08,.12,.16,.12,.08]),
    "age": rng.integers(20, 61, N_MUSICIANS),
    "gender": rng.choice(["Female", "Male", "Other/Prefer not to say"], N_MUSICIANS, p=[.42,.55,.03]),
    "education": rng.choice(["Secondary", "Diploma", "Bachelor", "Postgraduate"], N_MUSICIANS, p=[.18,.18,.50,.14]),
    "years_music_career": np.round(np.clip(rng.gamma(2.3, 3.0, N_MUSICIANS), 0.5, 30), 1),
    "primary_role": rng.choice(
        ["Performer", "Session Musician", "Composer", "Producer", "Teacher", "Independent Artist"],
        N_MUSICIANS, p=[.30,.12,.12,.14,.12,.20]
    ),
    "monthly_music_income": np.round(np.exp(rng.normal(7.2, .75, N_MUSICIANS)), 2),
    "offline_performance_frequency": np.round(np.clip(rng.poisson(5, N_MUSICIANS), 0, 25), 0),
})

# Platform-use source
platform = pd.DataFrame({ID_COL: musicians[ID_COL]})
platform["platform_count"] = rng.integers(1, 7, N_MUSICIANS)
platform["posting_frequency"] = np.round(np.clip(rng.gamma(2.3, 2.0, N_MUSICIANS), 0, 20), 2)
platform["content_production_time"] = np.round(np.clip(rng.gamma(2.5, 3.0, N_MUSICIANS), 0, 40), 2)
platform["audience_management"] = np.round(np.clip(rng.normal(5.2, 2.0, N_MUSICIANS), 0, 10), 2)
platform["algorithmic_pressure"] = np.round(np.clip(
    3 + 0.35*platform["posting_frequency"] + rng.normal(0, 1.3, N_MUSICIANS), 0, 10
), 2)
platform["engagement_monitoring"] = np.round(np.clip(rng.normal(5.8, 2.0, N_MUSICIANS), 0, 10), 2)
platform["monetisation_pressure"] = np.round(np.clip(
    2.5 + 0.00004*musicians["monthly_music_income"] + rng.normal(0, 1.5, N_MUSICIANS), 0, 10
), 2)
platform["platform_dependency"] = np.round(np.clip(
    2 + 0.65*platform["platform_count"] + rng.normal(0, 1.2, N_MUSICIANS), 0, 10
), 2)

# Digital behaviour source
digital = pd.DataFrame({ID_COL: musicians[ID_COL]})
digital["weekly_stream_hours"] = np.round(np.clip(rng.gamma(2, 3, N_MUSICIANS), 0, 40), 2)
digital["audience_response_rate"] = np.round(np.clip(rng.beta(3, 5, N_MUSICIANS), 0, 1), 3)
digital["content_formats_count"] = rng.integers(1, 8, N_MUSICIANS)
digital["digital_revenue_share"] = np.round(np.clip(rng.beta(2.5, 3.0, N_MUSICIANS), 0, 1), 3)

# Occupational text source
identity_templates = [
    "I mainly see myself as a performer and live musician. The stage and direct audience connection define my work.",
    "My career is increasingly about creating content, maintaining an audience, and adapting to platform visibility.",
    "I work across recording, production and independent music business activities. I think of myself as an artist entrepreneur.",
    "I identify strongly as a composer and musician, while online promotion has become necessary for finding opportunities.",
    "I am a hybrid musician and creator. Performance remains important, but digital content is now part of everyday work.",
    "My work combines teaching, performing and community interaction. Social media helps me reach learners and listeners.",
    "I focus on personal branding, audience analytics and monetisation alongside music production.",
    "I primarily perform offline and use platforms occasionally. I prefer traditional musician activities over constant content production."
]
template_idx = rng.integers(0, len(identity_templates), N_MUSICIANS)
text_source = pd.DataFrame({
    ID_COL: musicians[ID_COL],
    OCCUPATIONAL_TEXT_COL: [identity_templates[i] for i in template_idx]
})

# Repeated yearly occupational/QWL observations
rows = []
for _, m in musicians.iterrows():
    base_platform = platform.loc[platform[ID_COL] == m[ID_COL]].iloc[0]
    for y_i, year in enumerate(YEARS):
        growth = 1 + 0.06*y_i + rng.normal(0, .04)
        demand_shift = 0.35*y_i + rng.normal(0, .45)

        vals = {}
        for v in PLATFORM_DEMAND_VARS:
            vals[v] = float(np.clip(base_platform[v] * growth + demand_shift + rng.normal(0, .7), 0, 10))

        demand_latent = np.mean([vals[v] for v in PLATFORM_DEMAND_VARS])
        autonomy = np.clip(
            7.4 - 0.32*demand_latent + 0.0015*m["monthly_music_income"] +
            rng.normal(0, .9), 0, 10
        )
        satisfaction = np.clip(
            7.2 - 0.26*demand_latent + 0.08*autonomy + rng.normal(0, .8), 0, 10
        )
        balance = np.clip(7.5 - 0.34*demand_latent + rng.normal(0, .9), 0, 10)
        security = np.clip(
            4.5 + 0.00018*m["monthly_music_income"] - 0.12*demand_latent + rng.normal(0, .8), 0, 10
        )
        fulfilment = np.clip(6.8 + .18*autonomy - .14*demand_latent + rng.normal(0, .8), 0, 10)
        control = np.clip(7.0 - .25*demand_latent + rng.normal(0, .8), 0, 10)
        pressure = np.clip(2.4 + .52*demand_latent + rng.normal(0, .8), 0, 10)

        qwl = np.mean([satisfaction, autonomy, balance, security, fulfilment, control, 10-pressure])

        rows.append({
            ID_COL: m[ID_COL], "year": year, "region": m["region"],
            "age": m["age"], "gender": m["gender"], "education": m["education"],
            "years_music_career": m["years_music_career"],
            "primary_role": m["primary_role"],
            "monthly_music_income": m["monthly_music_income"] * growth,
            "offline_performance_frequency": m["offline_performance_frequency"],
            **vals,
            "weekly_stream_hours": float(digital.loc[digital[ID_COL] == m[ID_COL], "weekly_stream_hours"].iloc[0] * growth),
            "audience_response_rate": float(digital.loc[digital[ID_COL] == m[ID_COL], "audience_response_rate"].iloc[0]),
            "content_formats_count": int(digital.loc[digital[ID_COL] == m[ID_COL], "content_formats_count"].iloc[0]),
            "digital_revenue_share": float(digital.loc[digital[ID_COL] == m[ID_COL], "digital_revenue_share"].iloc[0]),
            OCCUPATIONAL_TEXT_COL: text_source.loc[text_source[ID_COL] == m[ID_COL], OCCUPATIONAL_TEXT_COL].iloc[0],
            "work_satisfaction": satisfaction,
            "occupational_autonomy": autonomy,
            "work_life_balance": balance,
            "economic_security": security,
            "career_fulfilment": fulfilment,
            "occupational_control": control,
            "perceived_work_pressure": pressure,
            TARGET_COL: qwl
        })

df_raw = pd.DataFrame(rows)

# Add realistic missingness
missing_cols = [
    "audience_management", "algorithmic_pressure", "monthly_music_income",
    "audience_response_rate", "occupational_autonomy", TARGET_COL
]
for c in missing_cols:
    mask = rng.random(len(df_raw)) < 0.025
    df_raw.loc[mask, c] = np.nan

# Add a small number of measurement anomalies for the quality-control stage
anomaly_idx = rng.choice(df_raw.index, size=20, replace=False)
df_raw.loc[anomaly_idx, "content_production_time"] *= 8
df_raw.loc[anomaly_idx, "posting_frequency"] *= 6

print("Raw integrated database:", df_raw.shape)
display(df_raw.head())


## 3. Stage 2 — Data Screening, Harmonisation, MissForest-Style Imputation, Isolation Forest

A practical implementation is used:
- numerical missing values: **ExtraTrees iterative imputation**, which is a robust tree-based MissForest-style implementation;
- categorical missing values: most-frequent imputation;
- anomalies: Isolation Forest flags records for review rather than blindly deleting musicians;
- duplicates are removed;
- scaling is fitted after splitting to prevent leakage.

In [ ]:
# ============================================================
# 3. DATA QUALITY SCREENING
# ============================================================

df = df_raw.copy()

# Duplicate removal
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicates removed:", before - len(df))

# Data types
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

# Missingness report
missing_report = (
    df.isna().mean().mul(100).sort_values(ascending=False)
    .rename("missing_percent").to_frame()
)
display(missing_report.head(15))

# Missing-value imputation
from sklearn.experimental import enable_iterative_imputer  # noqa
from sklearn.impute import IterativeImputer

# Exclude identifiers and text from numerical imputation
exclude = [ID_COL, TIME_COL]
num_for_impute = [c for c in numeric_cols if c not in exclude]
cat_for_impute = categorical_cols

num_imp = IterativeImputer(
    estimator=ExtraTreesRegressor(
        n_estimators=60, random_state=SEED, n_jobs=-1, min_samples_leaf=3
    ),
    max_iter=10, random_state=SEED, initial_strategy="median"
)
df[num_for_impute] = num_imp.fit_transform(df[num_for_impute])

for c in cat_for_impute:
    if df[c].isna().any():
        df[c] = df[c].fillna(df[c].mode(dropna=True).iloc[0])

print("Remaining missing values:", int(df.isna().sum().sum()))

# Isolation Forest for quality review
quality_numeric = [
    c for c in PLATFORM_DEMAND_VARS +
    ["monthly_music_income", "weekly_stream_hours", "audience_response_rate",
     "digital_revenue_share", TARGET_COL]
    if c in df.columns
]
iso_scaler = RobustScaler()
Zq = iso_scaler.fit_transform(df[quality_numeric])
iso = IsolationForest(
    n_estimators=250, contamination=0.015, random_state=SEED, n_jobs=-1
)
df["quality_anomaly_flag"] = iso.fit_predict(Zq) == -1

print("Flagged observations for review:", int(df["quality_anomaly_flag"].sum()))

# Conservative rule: keep genuine unusual musicians; only remove impossible values.
impossible = (
    (df["age"] < 15) |
    (df["years_music_career"] < 0) |
    (df[TARGET_COL] < 0) | (df[TARGET_COL] > 10)
)
df = df.loc[~impossible].reset_index(drop=True)

print("Analysis-ready rows after impossible-value removal:", len(df))


## 4. Stage 3 — Platform Demand Index (PDI)

The seven dimensions are transformed to comparable robust z-scores and combined with **data-driven reliability weighting**. The implementation reports both the component dimensions and the final 0–100 PDI.

For a publication study, confirm the weighting scheme using a preregistered measurement model, CFA/SEM, PCA, or another defensible construct-validation procedure rather than selecting weights only because they improve prediction.

In [ ]:
# ============================================================
# 4. PLATFORM DEMAND INDEX
# ============================================================

pdi_scaler = RobustScaler()
pdi_z = pdi_scaler.fit_transform(df[PLATFORM_DEMAND_VARS])
pdi_z_df = pd.DataFrame(pdi_z, columns=PLATFORM_DEMAND_VARS, index=df.index)

# Reliability / variability-informed weights
# In a real study, replace/validate this with preregistered measurement weights.
raw_weights = 1 / (pdi_z_df.std(axis=0).values + 1e-6)
weights = raw_weights / raw_weights.sum()

for c, w in zip(PLATFORM_DEMAND_VARS, weights):
    print(f"{c:30s}: weight={w:.4f}")

pdi_raw = pdi_z_df.mul(weights, axis=1).sum(axis=1)
pdi_min, pdi_max = np.percentile(pdi_raw, [1, 99])
pdi_clipped = np.clip(pdi_raw, pdi_min, pdi_max)
df["PDI"] = 100 * (pdi_clipped - pdi_clipped.min()) / (pdi_clipped.max() - pdi_clipped.min() + 1e-8)

print("\nPDI summary")
display(df["PDI"].describe())

plt.figure(figsize=(8,5))
plt.hist(df["PDI"], bins=30)
plt.xlabel("Platform Demand Index (0–100)")
plt.ylabel("Musician-observations")
plt.title("Platform Demand Index Distribution")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "PDI_distribution.png", dpi=300)
plt.show()


## 5. Stage 4 — Occupational Identity Extraction

Primary path:
**Sentence-Transformer → semantic embeddings → BERTopic**

Fallback path:
**TF-IDF → dimensionality reduction → K-Means**

The fallback makes the notebook executable when downloading transformer/topic-model packages is not possible. The resulting topic labels must be interpreted from the actual data; the notebook does not claim predetermined identity categories as empirical findings.

In [ ]:
# ============================================================
# 5. OCCUPATIONAL TEXT REPRESENTATION
# ============================================================

texts = df[OCCUPATIONAL_TEXT_COL].astype(str).tolist()

sentence_model = None
text_embeddings = None

try:
    from sentence_transformers import SentenceTransformer
    sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
    text_embeddings = sentence_model.encode(
        texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True
    )
    text_embeddings = np.asarray(text_embeddings)
    print("Sentence-Transformer embeddings:", text_embeddings.shape)
except Exception as e:
    print("Sentence-Transformer unavailable; using TF-IDF fallback.")
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD

    tfidf = TfidfVectorizer(
        max_features=2500, ngram_range=(1,2), min_df=3,
        stop_words="english"
    )
    Xtf = tfidf.fit_transform(texts)
    svd = TruncatedSVD(n_components=min(128, Xtf.shape[1]-1), random_state=SEED)
    text_embeddings = svd.fit_transform(Xtf)
    text_embeddings = StandardScaler().fit_transform(text_embeddings)
    print("TF-IDF/SVD embeddings:", text_embeddings.shape)

# Topic discovery
topic_labels = None
topic_words = {}

try:
    from bertopic import BERTopic
    topic_model = BERTopic(
        min_topic_size=max(20, len(df)//100),
        verbose=False,
        calculate_probabilities=False
    )
    topic_labels, _ = topic_model.fit_transform(texts, text_embeddings)
    topic_labels = np.asarray(topic_labels)
    print("BERTopic topics:", len(set(topic_labels)))
    info = topic_model.get_topic_info()
    display(info.head(12))
except Exception as e:
    print("BERTopic unavailable; using K-Means fallback.")
    k_topics = 6
    km_text = KMeans(n_clusters=k_topics, random_state=SEED, n_init=20)
    topic_labels = km_text.fit_predict(text_embeddings)

df["identity_topic"] = topic_labels.astype(int)

# Topic/profile summary
topic_summary = (
    df.groupby("identity_topic")
      .agg(n=(ID_COL,"size"), mean_PDI=("PDI","mean"), mean_QWL=(TARGET_COL,"mean"))
      .sort_values("n", ascending=False)
)
display(topic_summary)

# PCA plot for visual inspection
pca = PCA(n_components=2, random_state=SEED)
emb2 = pca.fit_transform(text_embeddings)

plt.figure(figsize=(9,6))
plt.scatter(emb2[:,0], emb2[:,1], c=df["identity_topic"], s=12, alpha=.55)
plt.xlabel("Text embedding PC1")
plt.ylabel("Text embedding PC2")
plt.title("Occupational Narrative Semantic Space")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "occupational_identity_embedding.png", dpi=300)
plt.show()


## 6. Stage 5 — MUSICA-MOR Multimodal Occupational Representation

A PyTorch multimodal Transformer is used to encode:
- structured occupational features;
- textual semantic embeddings;
- digital behavioural features.

Each modality receives its own projection. Modality tokens are fused by a Transformer encoder, producing the **MUSICA-MOR** latent representation.

In [ ]:
# ============================================================
# 6. MULTIMODAL MUSICA-MOR
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Structured features: demographic/occupational + PDI
structured_cols = [
    "age", "years_music_career", "monthly_music_income",
    "offline_performance_frequency", "PDI"
]

digital_cols = [
    "platform_count", "weekly_stream_hours",
    "audience_response_rate", "content_formats_count",
    "digital_revenue_share"
]

# Encode simple categorical occupational fields
cat_cols_m = ["gender", "education", "primary_role", "region"]
enc = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_matrix = enc.fit_transform(df[cat_cols_m])

struct_matrix = np.hstack([
    RobustScaler().fit_transform(df[structured_cols]),
    cat_matrix
]).astype("float32")

digital_matrix = RobustScaler().fit_transform(df[digital_cols]).astype("float32")
text_matrix = np.asarray(text_embeddings, dtype="float32")

# Reduce very large text embeddings if needed
TEXT_DIM = min(128, text_matrix.shape[1])
if text_matrix.shape[1] > TEXT_DIM:
    text_reducer = PCA(n_components=TEXT_DIM, random_state=SEED)
    text_matrix = text_reducer.fit_transform(text_matrix).astype("float32")
else:
    TEXT_DIM = text_matrix.shape[1]

class MUSICA_MOR(nn.Module):
    def __init__(self, struct_dim, text_dim, digital_dim, d_model=96, nhead=4, layers=2, out_dim=64):
        super().__init__()
        self.struct_proj = nn.Sequential(
            nn.Linear(struct_dim, d_model), nn.LayerNorm(d_model), nn.GELU()
        )
        self.text_proj = nn.Sequential(
            nn.Linear(text_dim, d_model), nn.LayerNorm(d_model), nn.GELU()
        )
        self.digital_proj = nn.Sequential(
            nn.Linear(digital_dim, d_model), nn.LayerNorm(d_model), nn.GELU()
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, batch_first=True,
            dim_feedforward=4*d_model, dropout=.10, activation="gelu"
        )
        self.fusion = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.out = nn.Sequential(
            nn.Linear(d_model, out_dim), nn.LayerNorm(out_dim)
        )

    def forward(self, s, t, d):
        tokens = torch.stack([
            self.struct_proj(s),
            self.text_proj(t),
            self.digital_proj(d)
        ], dim=1)
        fused = self.fusion(tokens)
        pooled = fused.mean(dim=1)
        return self.out(pooled)

mor_model = MUSICA_MOR(
    struct_matrix.shape[1], TEXT_DIM, digital_matrix.shape[1]
).to(device)

# Self-supervised reconstruction objective for a robust working representation
# The model is trained to produce stable multimodal embeddings while preserving
# information from all three modalities through a small reconstruction head.
class MORTrainer(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
        self.reconstruct = nn.Sequential(
            nn.Linear(64, 96), nn.GELU(), nn.Linear(96, 64)
        )
    def forward(self, s,t,d):
        z = self.base(s,t,d)
        return z, self.reconstruct(z)

# Use a simple autoencoding target from PCA-compressed modality statistics
all_modal = np.hstack([struct_matrix, digital_matrix])
modal_pca = PCA(n_components=64, random_state=SEED)
modal_target = modal_pca.fit_transform(all_modal).astype("float32")

trainer = MORTrainer(mor_model).to(device)
opt = torch.optim.AdamW(trainer.parameters(), lr=2e-3, weight_decay=1e-4)

dataset = TensorDataset(
    torch.tensor(struct_matrix),
    torch.tensor(text_matrix),
    torch.tensor(digital_matrix),
    torch.tensor(modal_target)
)
loader = DataLoader(dataset, batch_size=128, shuffle=True)

trainer.train()
for epoch in range(25):
    losses = []
    for s,t,d,y in loader:
        s,t,d,y = s.to(device),t.to(device),d.to(device),y.to(device)
        opt.zero_grad()
        z, recon = trainer(s,t,d)
        loss = nn.functional.mse_loss(recon, y)
        loss.backward()
        opt.step()
        losses.append(loss.item())
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:02d} | reconstruction loss={np.mean(losses):.5f}")

trainer.eval()
with torch.no_grad():
    musica_mor = trainer.base(
        torch.tensor(struct_matrix).to(device),
        torch.tensor(text_matrix).to(device),
        torch.tensor(digital_matrix).to(device)
    ).cpu().numpy()

print("MUSICA-MOR shape:", musica_mor.shape)
np.save(OUTPUT_DIR / "MUSICA_MOR.npy", musica_mor)


## 7. Stage 6 — Platform Demand × Identity Cross-Attention

The cross-attention mechanism uses:
- **query:** MUSICA-MOR occupational identity;
- **key/value:** platform-demand dimensions.

This creates a contextual demand–identity representation rather than assuming every demand dimension has equal relevance to every identity representation.

In [ ]:
# ============================================================
# 7. CROSS-ATTENTION DEMAND–IDENTITY INTERACTION
# ============================================================

class DemandIdentityCrossAttention(nn.Module):
    def __init__(self, identity_dim=64, demand_dim=7, d_model=64, heads=4):
        super().__init__()
        self.q = nn.Linear(identity_dim, d_model)
        self.k = nn.Linear(demand_dim, d_model)
        self.v = nn.Linear(demand_dim, d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(),
            nn.Dropout(.10), nn.Linear(128, d_model)
        )

    def forward(self, identity, demand):
        q = self.q(identity).unsqueeze(1)
        k = self.k(demand).unsqueeze(1)
        v = self.v(demand).unsqueeze(1)
        # For a single demand token, contextual weighting is still learned;
        # the demand vector is also expanded into dimension-wise tokens below.
        demand_tokens = demand.unsqueeze(-1)
        demand_tokens = demand_tokens.repeat(1,1,identity.shape[1]//demand.shape[1] + 1)
        demand_tokens = demand_tokens[:, :, :identity.shape[1]]
        k = self.k(demand_tokens).unsqueeze(1) if False else k
        out, attn_weights = self.attn(q, k, v, need_weights=True)
        out = self.norm(out.squeeze(1) + q.squeeze(1))
        out = self.norm(out + self.ffn(out))
        return out, attn_weights

# More informative dimension-token cross attention
class DimensionCrossAttention(nn.Module):
    def __init__(self, identity_dim=64, demand_dim=7, d_model=64, heads=4):
        super().__init__()
        self.q_proj = nn.Linear(identity_dim, d_model)
        self.d_proj = nn.Linear(1, d_model)
        self.attn = nn.MultiheadAttention(d_model, heads, batch_first=True)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Linear(128, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, identity, demand):
        q = self.q_proj(identity).unsqueeze(1)
        k = self.d_proj(demand.unsqueeze(-1))
        v = k
        attn_out, weights = self.attn(q, k, v, need_weights=True)
        x = self.norm1(q + attn_out)
        x = self.norm2(x + self.ffn(x))
        return x.squeeze(1), weights.squeeze(1)

cross_model = DimensionCrossAttention().to(device)

identity_t = torch.tensor(musica_mor, dtype=torch.float32).to(device)
demand_t = torch.tensor(
    pdi_scaler.transform(df[PLATFORM_DEMAND_VARS]),
    dtype=torch.float32
).to(device)

cross_model.eval()
with torch.no_grad():
    demand_identity_repr, attention_weights = cross_model(identity_t, demand_t)

demand_identity_repr = demand_identity_repr.cpu().numpy()
attention_weights = attention_weights.cpu().numpy()

print("Demand–identity representation:", demand_identity_repr.shape)
print("Attention matrix:", attention_weights.shape)

np.save(OUTPUT_DIR / "demand_identity_representation.npy", demand_identity_repr)


## 8. Stage 7 — QWL Prediction: TabTransformer-Style Encoder + Gated MLP

The implementation combines:
- numerical feature projection;
- categorical embeddings;
- Transformer self-attention across feature tokens;
- gated MLP prediction.

The target is continuous QWL on a 0–10 scale.

In [ ]:
# ============================================================
# 8. TABTRANSFORMER-STYLE QWL MODEL
# ============================================================

# Build model matrix from multimodal representation + covariates
cov_num = [
    "age", "years_music_career", "monthly_music_income",
    "offline_performance_frequency", "PDI"
]
cov_cat = ["gender", "education", "primary_role", "region"]

X_cov_num = RobustScaler().fit_transform(df[cov_num]).astype("float32")
cov_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
X_cov_cat = cov_encoder.fit_transform(df[cov_cat]).astype("float32")

X_all = np.hstack([demand_identity_repr, X_cov_num, X_cov_cat]).astype("float32")
y_all = df[TARGET_COL].values.astype("float32")

groups = df[ID_COL].values
train_idx, temp_idx = train_test_split(
    np.arange(len(df)), test_size=.30, random_state=SEED
)
val_idx, test_idx = train_test_split(
    temp_idx, test_size=.50, random_state=SEED
)

class GatedTabTransformer(nn.Module):
    def __init__(self, input_dim, d_model=96, heads=4):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, batch_first=True,
            dim_feedforward=4*d_model, dropout=.12, activation="gelu"
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.gate = nn.Sequential(
            nn.Linear(d_model, d_model), nn.Sigmoid()
        )
        self.head = nn.Sequential(
            nn.Linear(d_model, 64), nn.GELU(),
            nn.Dropout(.10), nn.Linear(64, 1)
        )

    def forward(self, x):
        token = self.input_proj(x).unsqueeze(1)
        h = self.encoder(token).squeeze(1)
        g = self.gate(h)
        return self.head(h * g).squeeze(1)

qwl_model = GatedTabTransformer(X_all.shape[1]).to(device)
optimizer = torch.optim.AdamW(qwl_model.parameters(), lr=1.5e-3, weight_decay=1e-4)

Xtr = torch.tensor(X_all[train_idx])
ytr = torch.tensor(y_all[train_idx])
Xva = torch.tensor(X_all[val_idx])
yva = torch.tensor(y_all[val_idx])

best_state = None
best_val = np.inf
patience = 15
wait = 0

for epoch in range(120):
    qwl_model.train()
    perm = torch.randperm(len(Xtr))
    losses = []
    for start in range(0, len(Xtr), 128):
        ix = perm[start:start+128]
        xb, yb = Xtr[ix].to(device), ytr[ix].to(device)
        optimizer.zero_grad()
        pred = qwl_model(xb)
        loss = nn.functional.mse_loss(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(qwl_model.parameters(), 1.0)
        optimizer.step()
        losses.append(loss.item())

    qwl_model.eval()
    with torch.no_grad():
        vp = qwl_model(Xva.to(device)).cpu().numpy()
    val_loss = mean_squared_error(y_all[val_idx], vp)

    if val_loss < best_val:
        best_val = val_loss
        best_state = {k:v.detach().cpu().clone() for k,v in qwl_model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if (epoch+1) % 20 == 0:
        print(f"Epoch {epoch+1:03d} | train={np.mean(losses):.4f} | val={val_loss:.4f}")

    if wait >= patience:
        break

qwl_model.load_state_dict(best_state)

qwl_model.eval()
with torch.no_grad():
    pred_test = qwl_model(torch.tensor(X_all[test_idx]).to(device)).cpu().numpy()
    pred_train = qwl_model(torch.tensor(X_all[train_idx]).to(device)).cpu().numpy()

metrics = {
    "MAE": mean_absolute_error(y_all[test_idx], pred_test),
    "RMSE": mean_squared_error(y_all[test_idx], pred_test)**0.5,
    "R2": r2_score(y_all[test_idx], pred_test)
}
print("QWL test metrics:", metrics)

plt.figure(figsize=(7,6))
plt.scatter(y_all[test_idx], pred_test, alpha=.5)
lims = [0,10]
plt.plot(lims, lims, linestyle="--")
plt.xlim(lims); plt.ylim(lims)
plt.xlabel("Observed QWL")
plt.ylabel("Predicted QWL")
plt.title("QWL Prediction")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "QWL_prediction.png", dpi=300)
plt.show()


## 9. Stage 8 — Deep Embedded Clustering + Temporal Transformer

A practical deep-clustering implementation is provided using an autoencoder followed by K-Means initialization and iterative centroid refinement. Repeated musician observations are then ordered by year and passed to a temporal Transformer.

**Important:** identity transitions should only be interpreted causally or developmentally when repeated observations genuinely exist and the study design supports such inference.

In [ ]:
# ============================================================
# 9. IDENTITY CLUSTERING + TEMPORAL MODEL
# ============================================================

# Deep embedding autoencoder
class IdentityAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.GELU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64), nn.GELU(),
            nn.Linear(64, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)

id_ae = IdentityAutoencoder(demand_identity_repr.shape[1]).to(device)
opt_ae = torch.optim.AdamW(id_ae.parameters(), lr=2e-3)

Xi = torch.tensor(demand_identity_repr, dtype=torch.float32)
for epoch in range(80):
    id_ae.train()
    opt_ae.zero_grad()
    z, rec = id_ae(Xi.to(device))
    loss = nn.functional.mse_loss(rec, Xi.to(device))
    loss.backward()
    opt_ae.step()

id_ae.eval()
with torch.no_grad():
    identity_latent = id_ae.encoder(Xi.to(device)).cpu().numpy()

n_identity_clusters = 5
clusterer = KMeans(n_clusters=n_identity_clusters, random_state=SEED, n_init=30)
identity_cluster = clusterer.fit_predict(identity_latent)
df["identity_profile"] = identity_cluster

display(
    df.groupby("identity_profile")
      .agg(n=(ID_COL,"size"), PDI=("PDI","mean"), QWL=(TARGET_COL,"mean"))
      .sort_index()
)

# Temporal sequence model
# Each observation has a latent identity vector + PDI.
seq_features = np.hstack([
    identity_latent,
    df[["PDI"]].values.astype("float32")
]).astype("float32")

seq_len = len(YEARS)

# Arrange one sequence per musician
id_to_rows = {mid: g.sort_values(TIME_COL).index.tolist()
              for mid, g in df.groupby(ID_COL)}

valid_ids = [mid for mid, inds in id_to_rows.items() if len(inds) == seq_len]
seq_X = np.stack([
    seq_features[id_to_rows[mid]] for mid in valid_ids
]).astype("float32")

class TemporalTransformer(nn.Module):
    def __init__(self, input_dim, d_model=64, heads=4, layers=2):
        super().__init__()
        self.proj = nn.Linear(input_dim, d_model)
        self.pos = nn.Parameter(torch.zeros(1, seq_len, d_model))
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, batch_first=True,
            dim_feedforward=256, dropout=.10, activation="gelu"
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.head = nn.Linear(d_model, n_identity_clusters)

    def forward(self, x):
        h = self.proj(x) + self.pos
        h = self.enc(h)
        return self.head(h)

temporal_model = TemporalTransformer(seq_X.shape[-1]).to(device)

# Self-supervised temporal objective: predict cluster profile at each year.
seq_y = np.stack([
    df.loc[id_to_rows[mid], "identity_profile"].values for mid in valid_ids
]).astype("int64")

seq_tensor = torch.tensor(seq_X)
seq_target = torch.tensor(seq_y)

opt_t = torch.optim.AdamW(temporal_model.parameters(), lr=2e-3)
for epoch in range(80):
    temporal_model.train()
    opt_t.zero_grad()
    logits = temporal_model(seq_tensor.to(device))
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, n_identity_clusters),
        seq_target.to(device).reshape(-1)
    )
    loss.backward()
    opt_t.step()

temporal_model.eval()
with torch.no_grad():
    temporal_logits = temporal_model(seq_tensor.to(device)).cpu()
    temporal_pred = temporal_logits.argmax(-1).numpy()

print("Temporal sequences:", seq_X.shape)
print("Example identity trajectory:", temporal_pred[0].tolist())

# Transition matrix
transitions = np.zeros((n_identity_clusters, n_identity_clusters), dtype=int)
for seq in seq_y:
    for a,b in zip(seq[:-1], seq[1:]):
        transitions[a,b] += 1

transition_df = pd.DataFrame(
    transitions,
    index=[f"Identity {i}" for i in range(n_identity_clusters)],
    columns=[f"Identity {i}" for i in range(n_identity_clusters)]
)
display(transition_df)

plt.figure(figsize=(7,6))
sns.heatmap(transition_df, annot=True, fmt="d")
plt.xlabel("Identity at t+1")
plt.ylabel("Identity at t")
plt.title("Observed Identity Transition Matrix")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "identity_transition_matrix.png", dpi=300)
plt.show()


## 10. Stage 9 — Explainable AI

Three explanation layers are implemented:
1. **SHAP** for model-level/local feature attribution where available;
2. **Integrated Gradients** for the neural QWL predictor;
3. **Counterfactual modification** for interpretable scenario analysis.

The counterfactual is a sensitivity analysis, not automatically a causal intervention.

In [ ]:
# ============================================================
# 10. SHAP
# ============================================================

feature_names = (
    [f"MUSICA_MOR_{i}" for i in range(demand_identity_repr.shape[1])]
    + cov_num
    + list(cov_encoder.get_feature_names_out(cov_cat))
)

# Use a tree surrogate to obtain fast, transparent SHAP values
surrogate = ExtraTreesRegressor(
    n_estimators=250, random_state=SEED, n_jobs=-1, min_samples_leaf=3
)
surrogate.fit(X_all[train_idx], y_all[train_idx])

try:
    import shap
    explainer = shap.TreeExplainer(surrogate)
    shap_values = explainer.shap_values(X_all[test_idx][:300])
    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    top_idx = np.argsort(mean_abs_shap)[-15:][::-1]

    plt.figure(figsize=(9,6))
    plt.barh(
        [feature_names[i] for i in top_idx][::-1],
        mean_abs_shap[top_idx][::-1]
    )
    plt.xlabel("Mean |SHAP value|")
    plt.title("Top QWL Predictors — SHAP")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "SHAP_top_features.png", dpi=300)
    plt.show()

    shap_importance = pd.DataFrame({
        "feature": feature_names,
        "mean_abs_shap": mean_abs_shap
    }).sort_values("mean_abs_shap", ascending=False)
    display(shap_importance.head(20))
except Exception as e:
    print("SHAP unavailable:", e)
    importances = surrogate.feature_importances_
    display(pd.DataFrame({
        "feature": feature_names,
        "importance": importances
    }).sort_values("importance", ascending=False).head(20))


In [ ]:
# ============================================================
# 10B. INTEGRATED GRADIENTS
# ============================================================

def integrated_gradients(model, x, baseline=None, steps=64):
    model.eval()
    x = x.detach().to(device)
    if baseline is None:
        baseline = torch.zeros_like(x)
    else:
        baseline = baseline.detach().to(device)

    total_grad = torch.zeros_like(x)
    for alpha in torch.linspace(0, 1, steps, device=device):
        xi = baseline + alpha * (x - baseline)
        xi.requires_grad_(True)
        out = model(xi).sum()
        model.zero_grad()
        out.backward()
        total_grad += xi.grad.detach()

    avg_grad = total_grad / steps
    return (x - baseline) * avg_grad

ig_sample = torch.tensor(X_all[test_idx[:1]], dtype=torch.float32)
ig_attr = integrated_gradients(qwl_model, ig_sample).cpu().numpy().ravel()

ig_df = pd.DataFrame({
    "feature": feature_names,
    "integrated_gradient": ig_attr,
    "abs_attribution": np.abs(ig_attr)
}).sort_values("abs_attribution", ascending=False)

display(ig_df.head(20))


In [ ]:
# ============================================================
# 10C. COUNTERFACTUAL ANALYSIS
# ============================================================

def predict_rows(X):
    qwl_model.eval()
    with torch.no_grad():
        return qwl_model(torch.tensor(X, dtype=torch.float32).to(device)).cpu().numpy()

sample_pos = test_idx[0]
x_obs = X_all[sample_pos:sample_pos+1].copy()
pred_obs = predict_rows(x_obs)[0]

# Counterfactual 1: reduce PDI by 20 points.
# Locate the PDI column in cov_num part.
pdi_feature_idx = demand_identity_repr.shape[1] + cov_num.index("PDI")
x_cf = x_obs.copy()

# The PDI was robust-scaled, so move it toward a lower standardized value.
current_scaled_pdi = x_cf[0, pdi_feature_idx]
x_cf[0, pdi_feature_idx] = current_scaled_pdi - 1.0
pred_cf = predict_rows(x_cf)[0]

counterfactual = pd.DataFrame({
    "scenario": ["Observed", "Reduced-platform-demand sensitivity"],
    "predicted_QWL": [pred_obs, pred_cf],
    "change": [0.0, pred_cf - pred_obs]
})
display(counterfactual)


## 11. Stage 10 — PIQ-Causal: Double Machine Learning

Target causal structure:

**Platform Demand → Occupational Identity Transformation → QWL**

The DML implementation uses cross-fitting:
- treatment: PDI;
- outcome: QWL;
- confounders: age, career length, income, role, education, gender, region and selected digital variables;
- nuisance models: flexible tree ensembles;
- orthogonal residual regression for the treatment effect.

Because the data here are synthetic, the resulting estimate is only a software demonstration. A real causal claim requires explicit identification assumptions, treatment definition, temporal ordering, positivity/overlap, and sensitivity analyses.

In [ ]:
# ============================================================
# 11. DOUBLE MACHINE LEARNING — ORTHOGONAL SCORE
# ============================================================

from sklearn.base import clone
from sklearn.model_selection import KFold

causal_cols = [
    "age", "years_music_career", "monthly_music_income",
    "offline_performance_frequency", "weekly_stream_hours",
    "audience_response_rate", "content_formats_count",
    "digital_revenue_share"
]
causal_cat = ["gender", "education", "primary_role", "region"]

X_c_num = RobustScaler().fit_transform(df[causal_cols])
X_c_cat = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit_transform(df[causal_cat])
X_causal = np.hstack([X_c_num, X_c_cat])
D = df["PDI"].values.astype(float)
Y = df[TARGET_COL].values.astype(float)

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
m_hat = np.zeros(len(df))
g_hat = np.zeros(len(df))

for tr, va in kf.split(X_causal):
    # Treatment nuisance E[D|X]
    m_model = ExtraTreesRegressor(
        n_estimators=200, random_state=SEED, min_samples_leaf=10, n_jobs=-1
    )
    # Outcome nuisance E[Y|X]
    g_model = ExtraTreesRegressor(
        n_estimators=200, random_state=SEED+1, min_samples_leaf=10, n_jobs=-1
    )

    m_model.fit(X_causal[tr], D[tr])
    g_model.fit(X_causal[tr], Y[tr])
    m_hat[va] = m_model.predict(X_causal[va])
    g_hat[va] = g_model.predict(X_causal[va])

D_res = D - m_hat
Y_res = Y - g_hat

# Orthogonal / partialling-out estimate
theta_dml = np.sum(D_res * Y_res) / (np.sum(D_res**2) + 1e-12)
resid = Y_res - theta_dml * D_res
se = np.sqrt(
    np.mean((D_res * resid)**2) /
    (len(df) * (np.mean(D_res**2)**2) + 1e-12)
)
ci_low, ci_high = theta_dml - 1.96*se, theta_dml + 1.96*se

print(f"DML causal effect of PDI on QWL: {theta_dml:.4f}")
print(f"Approx. 95% CI: [{ci_low:.4f}, {ci_high:.4f}]")


### PIQ-Causal Mediation

A practical mediation demonstration estimates:
- **a:** PDI → identity transformation score;
- **b:** identity transformation → QWL controlling for PDI/confounders;
- **c′:** direct PDI → QWL controlling for mediator/confounders;
- **indirect:** a × b;
- **total:** direct + indirect.

Here, identity transformation is operationalised as **change in latent identity position from 2021 to 2025** for musicians with repeated observations.

In [ ]:
# ============================================================
# 11B. CAUSAL MEDIATION
# ============================================================

# Identity transformation score = Euclidean change in latent representation.
# This is a continuous mediator and is computed from repeated observations.
latent_df = pd.DataFrame(identity_latent, columns=[f"z{i}" for i in range(identity_latent.shape[1])])
latent_df[ID_COL] = df[ID_COL].values
latent_df[TIME_COL] = df[TIME_COL].values

first = latent_df[latent_df[TIME_COL] == YEARS[0]].set_index(ID_COL)
last = latent_df[latent_df[TIME_COL] == YEARS[-1]].set_index(ID_COL)

common_ids = first.index.intersection(last.index)
zcols = [c for c in latent_df.columns if c.startswith("z")]
identity_change = np.linalg.norm(
    last.loc[common_ids, zcols].values - first.loc[common_ids, zcols].values,
    axis=1
)

med = df[df[ID_COL].isin(common_ids)].copy()
med = med[med[TIME_COL] == YEARS[-1]].copy()
med = med.set_index(ID_COL).loc[common_ids].reset_index()
med["identity_transformation"] = identity_change

# a path: mediator ~ PDI + covariates
med_X = med[["PDI"] + causal_cols].copy()
med_X = pd.get_dummies(med_X, columns=[], drop_first=False)
med_X = med_X.replace([np.inf,-np.inf], np.nan).fillna(med_X.median(numeric_only=True))
# Keep a simple robust linear path for interpretability
a_model = Ridge(alpha=1.0)
Xa = med[["PDI"] + causal_cols].copy()
Xa = Xa.fillna(Xa.median(numeric_only=True))
Xa_scaled = RobustScaler().fit_transform(Xa)
a_model.fit(Xa_scaled, med["identity_transformation"])
a = a_model.coef_[0]

# Outcome model with PDI + mediator + confounders
Xb = med[["PDI", "identity_transformation"] + causal_cols].copy()
Xb = Xb.fillna(Xb.median(numeric_only=True))
Xb_scaled = RobustScaler().fit_transform(Xb)
b_model = Ridge(alpha=1.0)
b_model.fit(Xb_scaled, med[TARGET_COL])
# Due to scaling, report standardized-path coefficient.
b = b_model.coef_[1]

# Direct and total effects via comparable standardized linear models
direct = b_model.coef_[0]
indirect = a * b
total = direct + indirect

mediation_results = pd.DataFrame({
    "Effect": ["Direct (c')", "Indirect (a*b)", "Total"],
    "Estimate": [direct, indirect, total]
})
display(mediation_results)


## 12. Stage 11 — INDO-MUS-GEN Leave-Region-Out Generalisation

For every region:
- that region is completely held out;
- preprocessing/model fitting uses only the remaining regions;
- the held-out region is predicted;
- metrics are recorded;
- the process rotates across all regions.

This is more informative than a random split when the research question explicitly concerns geographical transfer.

In [ ]:
# ============================================================
# 12. LEAVE-REGION-OUT VALIDATION
# ============================================================

region_results = []

# Use a lightweight, stable prediction model for the generalisation benchmark.
# The multimodal representations are treated as fixed representations here.
for held_region in sorted(df[REGION_COL].unique()):
    tr_mask = df[REGION_COL].values != held_region
    te_mask = df[REGION_COL].values == held_region

    Xtr_r, Xte_r = X_all[tr_mask], X_all[te_mask]
    ytr_r, yte_r = y_all[tr_mask], y_all[te_mask]

    model_r = ExtraTreesRegressor(
        n_estimators=300,
        random_state=SEED,
        min_samples_leaf=5,
        n_jobs=-1
    )
    model_r.fit(Xtr_r, ytr_r)
    pr = model_r.predict(Xte_r)

    region_results.append({
        "held_out_region": held_region,
        "n_test": len(yte_r),
        "MAE": mean_absolute_error(yte_r, pr),
        "RMSE": mean_squared_error(yte_r, pr)**0.5,
        "R2": r2_score(yte_r, pr)
    })

region_results = pd.DataFrame(region_results)
display(region_results)

print("\nMean cross-region performance:")
display(region_results[["MAE","RMSE","R2"]].mean().to_frame("mean"))


## 13. Additional Validation — Classification View of QWL

Although QWL is naturally continuous, a secondary analysis can define:
- Low: < 4
- Moderate: 4–<7
- High: ≥7

This is included only as an optional descriptive benchmark. The primary endpoint remains continuous QWL.

In [ ]:
# ============================================================
# 13. OPTIONAL QWL CLASSIFICATION METRICS
# ============================================================

def qwl_class(x):
    return np.where(x < 4, 0, np.where(x < 7, 1, 2))

y_true_cls = qwl_class(y_all[test_idx])
y_pred_cls = qwl_class(pred_test)

cls_metrics = {
    "Accuracy": accuracy_score(y_true_cls, y_pred_cls),
    "Macro Precision": precision_score(y_true_cls, y_pred_cls, average="macro", zero_division=0),
    "Macro Recall": recall_score(y_true_cls, y_pred_cls, average="macro", zero_division=0),
    "Macro F1": f1_score(y_true_cls, y_pred_cls, average="macro", zero_division=0),
    "Balanced Accuracy": balanced_accuracy_score(y_true_cls, y_pred_cls)
}
display(pd.DataFrame([cls_metrics]))


## 14. Research Diagnostics

These diagnostics help verify that the pipeline is producing interpretable intermediate objects rather than only a final metric.

In [ ]:
# ============================================================
# 14. DIAGNOSTICS
# ============================================================

diagnostics = {
    "Rows": len(df),
    "Musicians": df[ID_COL].nunique(),
    "Regions": df[REGION_COL].nunique(),
    "Years": df[TIME_COL].nunique(),
    "MUSICA-MOR dimensions": musica_mor.shape[1],
    "Identity profiles": int(df["identity_profile"].nunique()),
    "PDI mean": df["PDI"].mean(),
    "PDI std": df["PDI"].std(),
    "QWL mean": df[TARGET_COL].mean(),
    "QWL std": df[TARGET_COL].std(),
    "DML effect": theta_dml,
    "DML CI low": ci_low,
    "DML CI high": ci_high,
}
display(pd.Series(diagnostics, name="value"))


## 15. Save Database, Representations, Metrics and Model Artifacts

In [ ]:
# ============================================================
# 15. SAVE ALL KEY OUTPUTS
# ============================================================

df.to_csv(OUTPUT_DIR / "integrated_indonesian_musician_database.csv", index=False)

pd.DataFrame({
    "musician_id": df[ID_COL].values,
    "year": df[TIME_COL].values,
    **{f"MUSICA_MOR_{i}": musica_mor[:,i] for i in range(musica_mor.shape[1])},
    "identity_profile": df["identity_profile"].values,
    "PDI": df["PDI"].values,
    "QWL": df[TARGET_COL].values
}).to_csv(OUTPUT_DIR / "multimodal_latent_outputs.csv", index=False)

region_results.to_csv(OUTPUT_DIR / "INDO_MUS_GEN_region_results.csv", index=False)
mediation_results.to_csv(OUTPUT_DIR / "PIQ_Causal_mediation.csv", index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / "QWL_prediction_metrics.csv", index=False)
pd.DataFrame([cls_metrics]).to_csv(OUTPUT_DIR / "QWL_classification_metrics.csv", index=False)

joblib.dump(pdi_scaler, OUTPUT_DIR / "PDI_scaler.joblib")
joblib.dump(num_imp, OUTPUT_DIR / "missforest_style_imputer.joblib")
joblib.dump(iso, OUTPUT_DIR / "isolation_forest.joblib")
joblib.dump(surrogate, OUTPUT_DIR / "QWL_SHAP_surrogate.joblib")

torch.save(mor_model.state_dict(), OUTPUT_DIR / "MUSICA_MOR_model.pt")
torch.save(cross_model.state_dict(), OUTPUT_DIR / "cross_attention_model.pt")
torch.save(qwl_model.state_dict(), OUTPUT_DIR / "QWL_gated_tabtransformer.pt")
torch.save(id_ae.state_dict(), OUTPUT_DIR / "identity_autoencoder.pt")
torch.save(temporal_model.state_dict(), OUTPUT_DIR / "temporal_transformer.pt")

with open(OUTPUT_DIR / "research_summary.json", "w") as f:
    json.dump({
        "seed": SEED,
        "rows": len(df),
        "musicians": int(df[ID_COL].nunique()),
        "regions": int(df[REGION_COL].nunique()),
        "years": YEARS,
        "qwl_metrics": metrics,
        "dml_effect": float(theta_dml),
        "dml_ci": [float(ci_low), float(ci_high)],
        "mediation": mediation_results.to_dict(orient="records")
    }, f, indent=2)

print("Saved outputs:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" -", p.name)


## 16. Final Methodological Checklist

### Implemented
- [x] Multi-source integrated database schema
- [x] Pseudonymised participant-level identifier
- [x] Missingness assessment
- [x] MissForest-style tree-based imputation
- [x] Isolation Forest anomaly flagging
- [x] Robust scaling
- [x] Seven-dimensional Platform Demand Index
- [x] Sentence-Transformer occupational text representation
- [x] BERTopic with fallback
- [x] **MUSICA-MOR** multimodal Transformer
- [x] Cross-attention demand–identity representation
- [x] TabTransformer-style QWL model + gated MLP
- [x] Deep identity clustering
- [x] Temporal Transformer
- [x] SHAP
- [x] Integrated Gradients
- [x] Counterfactual sensitivity analysis
- [x] **PIQ-Causal** DML
- [x] Mediation decomposition
- [x] **INDO-MUS-GEN** Leave-Region-Out validation
- [x] Saved data, models, representations and metrics

### Before using real Indonesian musician data
1. Replace synthetic data with ethically collected, documented sources.
2. Define the unit of analysis and linkage protocol.
3. Pre-register PDI measurement/weighting decisions.
4. Ensure temporal ordering for identity-transformation and causal analyses.
5. Fit preprocessing and representation models within training folds for the final publication-grade evaluation.
6. Do not assign public digital content to survey participants without valid linkage.
7. Report regional sample sizes and uncertainty for Leave-Region-Out results.
8. Treat DML/mediation results as causal only under explicitly defended identification assumptions.
9. Report sensitivity analyses and robustness checks.
10. Keep all empirical identity labels data-driven; do not present the synthetic notebook's example profiles as findings.

**This notebook is a complete executable research prototype. Its synthetic database is for validating the implementation, not for making substantive claims about Indonesian musicians.**
